# 100 Days of AI — Wav2Lip Talking Head Pipeline

**What this does:** Portrait image + MP3 audio → a lip-synced talking-head video. Much faster than SadTalker — minutes, not hours (it only repaints the mouth region instead of rendering a whole animated head).

**Setup — run once per Colab session:** cells 1–5 (GPU → clone → patches → weights → Drive → avatar).

**Per lesson:** run Cell 6 → 7 → 8, changing the audio upload and the `LESSON` value each time. No need to re-run setup between lessons.

> Runtime → Change runtime type → **T4 GPU**

## Section 1 — Setup (run once per session)

In [ ]:
# Cell 1: Check GPU
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
else:
    raise RuntimeError('No GPU detected. Go to Runtime → Change runtime type → T4 GPU.')

In [ ]:
# Cell 2: Clone Wav2Lip and install dependencies
import os

# Pin NumPy to 1.x — Wav2Lip's audio stack is not compatible with NumPy 2.0
!pip install -q "numpy<2.0"

if not os.path.exists('/content/Wav2Lip'):
    !git clone -q https://github.com/justinjohn0306/Wav2Lip.git /content/Wav2Lip
else:
    print('Wav2Lip already cloned — skipping.')

%cd /content/Wav2Lip
# Modern, Colab-compatible deps (the repo's pinned requirements.txt is too old for Python 3.12).
# batch-face = the face detector this fork's inference.py imports (RetinaFace).
!pip install -q "numpy<2.0" librosa==0.10.2.post1 opencv-python numba tqdm batch-face
print('Done.')

In [ ]:
# Cell 2b: Compatibility patches for modern Colab (librosa + numpy)

# 1. audio.py calls librosa.filters.mel with positional args — newer librosa requires keywords
!sed -i 's/librosa.filters.mel(hp.sample_rate, hp.n_fft,/librosa.filters.mel(sr=hp.sample_rate, n_fft=hp.n_fft,/' /content/Wav2Lip/audio.py

# 2. librosa.core.* was removed in newer librosa — rewrite to top-level librosa.*
!sed -i 's/librosa\.core\./librosa./g' /content/Wav2Lip/audio.py

# 3. NumPy removed np.float / np.int / np.bool aliases — patch every .py file in one pass
!find /content/Wav2Lip -name "*.py" -exec sed -i \
    's/np\.float\b/float/g; s/np\.int\b/int/g; s/np\.bool\b/bool/g' {} \;

print('Patches applied.')

In [ ]:
# Cell 3: Download model weights
# All from the fork's GitHub releases (stable links):
#   wav2lip_gan.pth = the lip-sync model
#   mobilenet.pth   = the RetinaFace face detector this fork's inference.py loads
import os

os.makedirs('/content/Wav2Lip/checkpoints', exist_ok=True)

GAN = '/content/Wav2Lip/checkpoints/wav2lip_gan.pth'
DET = '/content/Wav2Lip/checkpoints/mobilenet.pth'

if os.path.exists(GAN) and os.path.exists(DET):
    print('Weights already present — skipping download.')
else:
    !wget -q --show-progress 'https://github.com/justinjohn0306/Wav2Lip/releases/download/models/wav2lip_gan.pth' -O "{GAN}"
    !wget -q --show-progress 'https://github.com/justinjohn0306/Wav2Lip/releases/download/models/mobilenet.pth' -O "{DET}"
    print('Weights downloaded.')

## Section 2 — Files (portrait + audio)

In [ ]:
# Cell 4: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/100DaysOfAI'
os.makedirs(f'{DRIVE_DIR}/audio', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/videos', exist_ok=True)
print(f'Drive mounted. Working folder: {DRIVE_DIR}')

In [ ]:
# Cell 5: Set up portrait — upload Avatar.png on first use, reuse from Drive after that
import os
from google.colab import files

AVATAR_DRIVE = '/content/drive/MyDrive/100DaysOfAI/Avatar.png'
AVATAR_LOCAL = '/content/avatar.png'

if os.path.exists(AVATAR_DRIVE):
    !cp "{AVATAR_DRIVE}" "{AVATAR_LOCAL}"
    print('Avatar loaded from Drive.')
else:
    print('Avatar not found in Drive. Upload Avatar.png now:')
    uploaded = files.upload()  # upload Avatar.png
    fname = list(uploaded.keys())[0]
    !mv "{fname}" "{AVATAR_LOCAL}"
    !cp "{AVATAR_LOCAL}" "{AVATAR_DRIVE}"
    print(f'Avatar saved to Drive for future sessions.')

from IPython.display import Image, display
display(Image(AVATAR_LOCAL, width=200))

In [ ]:
# Cell 6: Upload today's audio file
# Filename must match your local pipeline output, e.g. day_001_lesson_02.mp3
from google.colab import files
import shutil, os

print('Upload the MP3 from 00_pipeline/audio/ on your machine:')
uploaded = files.upload()
fname = list(uploaded.keys())[0]

AUDIO_LOCAL = f'/content/{fname}'
if not os.path.exists(AUDIO_LOCAL):
    shutil.move(fname, AUDIO_LOCAL)

# Mirror to Drive so it's saved
shutil.copy(AUDIO_LOCAL, f'/content/drive/MyDrive/100DaysOfAI/audio/{fname}')
print(f'Audio ready: {AUDIO_LOCAL}')

## Section 3 — Generate

In [ ]:
# Cell 7: Run Wav2Lip
# ── Change LESSON to match the audio you uploaded (no .mp3) ──
LESSON = 'day_001_lesson_01'
# ─────────────────────────────────────────────────────────────

import os

AUDIO = f'/content/{LESSON}.mp3'
FACE  = '/content/avatar.png'
OUT   = f'/content/results/{LESSON}_talking_head.mp4'

os.makedirs('/content/results', exist_ok=True)

if not os.path.exists(AUDIO):
    raise FileNotFoundError(f'Audio not found: {AUDIO} — did you upload {LESSON}.mp3 in Cell 6?')

%cd /content/Wav2Lip

!python inference.py \
    --checkpoint_path checkpoints/wav2lip_gan.pth \
    --face "{FACE}" \
    --audio "{AUDIO}" \
    --outfile "{OUT}" \
    --pads 0 15 0 0 \
    --fps 25 \
    --nosmooth \
    --resize_factor 1

# Confirm it actually produced the file (a failed !python above does NOT stop the cell)
if os.path.exists(OUT):
    print('\nWav2Lip done →', OUT)
else:
    raise RuntimeError('Wav2Lip did not produce an output — read the error in the log above.')

In [ ]:
# Cell 8: Save output video to Google Drive
import shutil, os

OUT  = f'/content/results/{LESSON}_talking_head.mp4'
dest = f'/content/drive/MyDrive/100DaysOfAI/videos/{LESSON}_talking_head.mp4'

if not os.path.exists(OUT):
    raise FileNotFoundError(f'No output at {OUT} — check the Wav2Lip output above.')

shutil.copy(OUT, dest)
print(f'Saved to Drive: {dest}')

# Preview
from IPython.display import Video
Video(OUT, width=400)